# MIR — Benchmark d'indexation des descripteurs sur GPU T4 (Google Colab)

Ce notebook mesure le **temps réel d'indexation** de tous les descripteurs du projet MIR sur le GPU T4 de Colab :
- **Descripteurs classiques** : Histogramme de couleur, SIFT (CPU)
- **Descripteurs deep learning** : MobileNetV2, ResNet50, ViT-B/16, DinoV2 (GPU)

Pipeline de mesure pour chaque descripteur :
1. Prétraitement des images
2. Extraction des descripteurs (inférence batch sur T4)
3. Réduction PCA + whitening + normalisation L2
4. Temps total = extraction + PCA

Les résultats sont sauvegardés dans `Drive/MIR_Project/indexing_benchmark/` (JSON + graphiques).

---
> **Prérequis** : Activer le GPU T4 dans `Exécution > Modifier le type d'exécution > T4 GPU`

In [ ]:
# ── Installation des dépendances ──────────────────────────────────────────────
!pip install timm==1.0.15 tqdm opencv-python-headless --quiet

In [ ]:
# ── Montage Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Configuration — MODIFIER SI NÉCESSAIRE ───────────────────────────────────
# Chemin racine du projet sur Drive
DRIVE_ROOT  = "/content/drive/MyDrive/MIR_Project"

# Dossier contenant les images du dataset (*.jpg)
DATASET_DIR = f"{DRIVE_ROOT}/dataset"

# Dossier contenant les poids fine-tunés (.pth)
MODELS_DIR  = f"{DRIVE_ROOT}/modelsV2"

# Dossier de sortie pour les résultats (créé automatiquement)
SAVE_DIR    = f"{DRIVE_ROOT}/indexing_benchmark"

# Noms des fichiers de poids fine-tunés dans MODELS_DIR
FINETUNED_WEIGHTS = {
    "mobilenet_arcface": "Copie de MobileNetV2_ARCFace_finetuned_best.pth",
    "dinov2_supcon":     "Copie de DinoV2_SupCon_finetuned_best.pth",
}
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import os, time, json, glob, warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
if device.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ Aucun GPU détecté — activer le T4 dans les paramètres d'exécution")

print(f"Save dir: {SAVE_DIR}")

In [ ]:
# ── Classe MetricModel (identique à app/training/models.py) ──────────────────
# Utilisée pour charger les poids fine-tunés MobileNet-ArcFace et DinoV2-SupCon

EMBED_DIM = 512

MODEL_CONFIGS = {
    "mobilenetv2": {"timm_name": "mobilenetv2_100.ra_in1k",              "img_size": 224, "feat_dim": 1280},
    "resnet50":    {"timm_name": "resnet50.a1_in1k",                      "img_size": 224, "feat_dim": 2048},
    "vit_base":    {"timm_name": "vit_base_patch16_224.augreg_in1k",      "img_size": 224, "feat_dim": 768},
    "dinov2":      {"timm_name": "vit_small_patch14_dinov2.lvd142m",      "img_size": 518, "feat_dim": 384},
}

class MetricModel(nn.Module):
    """Backbone timm partiellement fine-tuné + tête de projection → 512-dim."""
    def __init__(self, model_name: str, img_size: int = None):
        super().__init__()
        cfg = MODEL_CONFIGS[model_name]
        self.model_name = model_name
        self.img_size   = img_size or cfg["img_size"]
        is_vit = model_name in ("vit_base", "dinov2")
        self.backbone = timm.create_model(
            cfg["timm_name"], pretrained=False, num_classes=0,
            **(({"img_size": self.img_size}) if is_vit else {}),
        )
        feat_dim = cfg["feat_dim"]
        self.projection = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, EMBED_DIM),
        )

    def forward(self, x):
        feats = self.backbone(x).float()
        return F.normalize(self.projection(feats), p=2, dim=1)


def load_metric_model(model_name: str, weights_path: str):
    """Charge un MetricModel depuis un fichier .pth. Retourne None si incompatible."""
    try:
        state = torch.load(weights_path, map_location="cpu", weights_only=False)
        img_size = state.get("hyperparams", {}).get("img_size")
        model = MetricModel(model_name, img_size=img_size)
        # Supporte model_state_dict ou model_state
        sd = state.get("model_state_dict") or state.get("model_state")
        if sd is None:
            return None
        model.load_state_dict(sd, strict=False)
        model.eval().float()
        return model
    except Exception as e:
        print(f"  ⚠ Chargement MetricModel échoué ({e}) — fallback pretrained timm")
        return None


print("MetricModel défini.")

In [ ]:
# ── Helpers : prétraitement, extraction, PCA ──────────────────────────────────

_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
EPS   = 1e-3  # régularisation whitening


def preprocess_batch(paths, img_size):
    """Charge et prétraite une liste d'images → tenseur (N, 3, H, W)."""
    tensors = []
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            tensors.append(torch.zeros(3, img_size, img_size))
            continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pil = Image.fromarray(rgb).resize((img_size, img_size), Image.BILINEAR)
        arr = (np.array(pil, dtype=np.float32) / 255.0 - _MEAN) / _STD
        tensors.append(torch.from_numpy(arr).permute(2, 0, 1))
    return torch.stack(tensors)


def extract_embeddings_gpu(model, image_paths, img_size, batch_size, desc=""):
    """Inférence batch sur GPU. Retourne (N, D) numpy float32."""
    model = model.to(device)
    all_feats = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc=f"  {desc}", ncols=80):
        batch = preprocess_batch(image_paths[i:i+batch_size], img_size).to(device)
        with torch.no_grad():
            feats = model(batch).float().cpu().numpy()
        all_feats.append(feats)
    return np.concatenate(all_feats, axis=0)


def fit_pca_whitening(X, k):
    """PCA + whitening sur X (n×d). Retourne (mean, composantes k×d, échelle k)."""
    mean = X.mean(axis=0)
    Xc   = X - mean
    C    = (Xc.T @ Xc) / len(Xc)
    eigenvalues, eigenvectors = np.linalg.eigh(C)
    idx = np.argsort(eigenvalues)[::-1]
    components      = eigenvectors[:, idx][:, :k].T.astype(np.float32)  # (k, d)
    whitening_scale = (1.0 / np.sqrt(eigenvalues[idx][:k] + EPS)).astype(np.float32)
    return mean.astype(np.float32), components, whitening_scale


def apply_pca_whitening(X, mean, components, whitening_scale):
    """Projette, blanchit et normalise L2."""
    Xw = (X - mean) @ components.T * whitening_scale
    norms = np.linalg.norm(Xw, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return (Xw / norms).astype(np.float32)


def format_time(s):
    if s >= 3600: return f"{int(s//3600)}h {int((s%3600)//60)}m"
    if s >= 60:   return f"{s/60:.1f} min"
    return f"{s:.1f} s"


print("Helpers définis.")

In [ ]:
# ── Chargement de la liste d'images ──────────────────────────────────────────
image_paths = sorted(glob.glob(os.path.join(DATASET_DIR, "*.jpg")))

if not image_paths:
    raise FileNotFoundError(f"Aucune image .jpg trouvée dans {DATASET_DIR}")

print(f"Dataset : {len(image_paths)} images")
print(f"Exemple : {os.path.basename(image_paths[0])}")
N_IMAGES = len(image_paths)

# Stockage des résultats de timing
timing_results = {}

## 1. Descripteurs classiques (CPU)

- **Histogramme de couleur** : histogramme RGB 8 bins/canal, normalisé L1 — 24 dimensions
- **SIFT** : moyenne des descripteurs SIFT détectés, normalisé L2 — 128 dimensions

In [ ]:
# ── Histogramme de couleur ────────────────────────────────────────────────────
print("[color_histogram] Démarrage...")

features_hist = []
t_start = time.perf_counter()

for path in tqdm(image_paths, desc="  color_histogram", ncols=80):
    img = cv2.imread(path)
    if img is None:
        features_hist.append(np.zeros(24, dtype=np.float32))
        continue
    bins = 8
    hist_r = cv2.calcHist([img], [0], None, [bins], [0, 256]).flatten()
    hist_g = cv2.calcHist([img], [1], None, [bins], [0, 256]).flatten()
    hist_b = cv2.calcHist([img], [2], None, [bins], [0, 256]).flatten()
    feat = np.concatenate([hist_r, hist_g, hist_b]).astype(np.float32)
    s = feat.sum()
    features_hist.append(feat / s if s > 0 else feat)

indexing_time = time.perf_counter() - t_start

# Pas de PCA pour l'histogramme couleur
index_matrix = np.array(features_hist)
avg_search = indexing_time / N_IMAGES

timing_results["color_histogram"] = {
    "indexing_time_s": round(indexing_time, 3),
    "extraction_time_s": round(indexing_time, 3),
    "pca_time_s": 0.0,
    "descriptor_dim": 24,
    "original_dim": None,
    "num_images": N_IMAGES,
    "device": "cpu",
}

print(f"  Temps total : {format_time(indexing_time)}  ({indexing_time/N_IMAGES*1000:.2f} ms/img)")
print(f"  Résultat    : {index_matrix.shape}")

In [ ]:
# ── SIFT ──────────────────────────────────────────────────────────────────────
print("[sift] Démarrage...")

_sift = cv2.SIFT_create(nfeatures=500)
features_sift = []
t_start = time.perf_counter()

for path in tqdm(image_paths, desc="  sift", ncols=80):
    img = cv2.imread(path)
    if img is None:
        features_sift.append(np.zeros(128, dtype=np.float32))
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, descriptors = _sift.detectAndCompute(gray, None)
    if descriptors is None or len(descriptors) == 0:
        features_sift.append(np.zeros(128, dtype=np.float32))
        continue
    feat = descriptors.mean(axis=0).astype(np.float32)
    norm = np.linalg.norm(feat)
    features_sift.append(feat / norm if norm > 0 else feat)

indexing_time = time.perf_counter() - t_start

index_matrix = np.array(features_sift)

timing_results["sift"] = {
    "indexing_time_s": round(indexing_time, 3),
    "extraction_time_s": round(indexing_time, 3),
    "pca_time_s": 0.0,
    "descriptor_dim": 128,
    "original_dim": None,
    "num_images": N_IMAGES,
    "device": "cpu",
}

print(f"  Temps total : {format_time(indexing_time)}  ({indexing_time/N_IMAGES*1000:.2f} ms/img)")
print(f"  Résultat    : {index_matrix.shape}")

## 2. Descripteurs Deep Learning (GPU T4)

Pipeline pour chaque descripteur DL :
1. Chargement du modèle (fine-tuné ou pretrained timm)
2. Inférence batch sur GPU T4 → embeddings bruts
3. Ajustement PCA + whitening depuis zéro (comme lors d'une vraie indexation)
4. Transformation PCA + normalisation L2

Les modèles partageant le même backbone (MobileNetV2 → arcface + zeroshot ; DinoV2 → supcon + zeroshot) sont regroupés pour ne faire qu'**un seul chargement** du backbone.

| Descripteur | Backbone | Dim brute → PCA |
|---|---|---|
| mobilenet_arcface | MobileNetV2 fine-tuné | 512 → 49 |
| mobilenet_zeroshot | MobileNetV2 pretrained | 1280 → 257 |
| resnet50_zeroshot | ResNet50 pretrained | 2048 → 256 |
| vit_b16_zeroshot | ViT-B/16 pretrained | 768 → 230 |
| dinov2_supcon | DinoV2 fine-tuné | 512 → 13 |
| dinov2_zeroshot | DinoV2 pretrained | 384 → 13 |

In [ ]:
# ── Configuration des descripteurs DL ────────────────────────────────────────

# Dimensions PCA cibles (95% de variance, identiques au projet)
PCA_DIMS = {
    "mobilenet_arcface":  49,
    "mobilenet_zeroshot": 257,
    "resnet50_zeroshot":  256,
    "vit_b16_zeroshot":   230,
    "dinov2_supcon":      13,
    "dinov2_zeroshot":    13,
}

# Groupes de descripteurs par backbone (ordre : rapide → lent)
# Pour chaque groupe : (backbone_name, timm_name, img_size, batch_size, descripteurs)
BACKBONE_GROUPS = [
    {
        "backbone":    "mobilenetv2",
        "timm_name":  "mobilenetv2_100.ra_in1k",
        "img_size":   224,
        "batch_size": 256,
        "descriptors": [
            # (nom, fichier_poids_finetune_ou_None)
            ("mobilenet_arcface",  FINETUNED_WEIGHTS.get("mobilenet_arcface")),
            ("mobilenet_zeroshot", None),
        ],
    },
    {
        "backbone":    "resnet50",
        "timm_name":  "resnet50.a1_in1k",
        "img_size":   224,
        "batch_size": 128,
        "descriptors": [
            ("resnet50_zeroshot", None),
        ],
    },
    {
        "backbone":    "vit_base",
        "timm_name":  "vit_base_patch16_224.augreg_in1k",
        "img_size":   224,
        "batch_size": 64,
        "descriptors": [
            ("vit_b16_zeroshot", None),
        ],
    },
    {
        "backbone":    "dinov2",
        "timm_name":  "vit_small_patch14_dinov2.lvd142m",
        "img_size":   518,
        "batch_size": 32,
        "descriptors": [
            ("dinov2_supcon",    FINETUNED_WEIGHTS.get("dinov2_supcon")),
            ("dinov2_zeroshot", None),
        ],
    },
]

print("Configuration DL définie.")

In [ ]:
# ── Mesure des temps d'indexation DL ─────────────────────────────────────────
# Pour chaque groupe de backbone :
#   1. Extraction backbone pretrained (partagée entre tous les descripteurs du groupe)
#   2. Pour les descripteurs fine-tunés : extraction avec MetricModel (tête de projection)
#   3. PCA individuelle par descripteur (dimensions cibles différentes)

for group in BACKBONE_GROUPS:
    backbone_name = group["backbone"]
    timm_name     = group["timm_name"]
    img_size      = group["img_size"]
    batch_size    = group["batch_size"]
    descriptors   = group["descriptors"]

    print(f"\n{'='*60}")
    print(f"Backbone : {backbone_name}  ({timm_name})")
    print(f"Images   : {N_IMAGES}  |  batch={batch_size}  |  size={img_size}px")
    print(f"{'='*60}")

    # ── Prétraitement + extraction backbone pretrained (partagé) ──────────────
    print("Chargement backbone pretrained (timm)...")
    backbone = timm.create_model(timm_name, pretrained=True, num_classes=0)
    backbone.eval().float().to(device)

    t_extract_start = time.perf_counter()
    base_embeddings = extract_embeddings_gpu(
        backbone, image_paths, img_size, batch_size, desc=backbone_name
    )
    t_extract_end   = time.perf_counter()
    extract_time    = t_extract_end - t_extract_start

    print(f"  Extraction backbone : {format_time(extract_time)}  "
          f"({extract_time/N_IMAGES*1000:.2f} ms/img)  "
          f"→ shape {base_embeddings.shape}")

    # ── Traitement par descripteur ─────────────────────────────────────────────
    for desc_name, weights_file in descriptors:
        pca_k = PCA_DIMS[desc_name]
        print(f"\n  [{desc_name}]  PCA cible : {pca_k}D")

        # Décide si on utilise les embeddings backbone bruts
        # ou ceux d'un MetricModel fine-tuné
        embeddings  = base_embeddings
        ft_time     = 0.0
        model_type  = "pretrained"

        if weights_file is not None:
            weights_path = os.path.join(MODELS_DIR, weights_file)
            if os.path.exists(weights_path):
                print(f"  Chargement poids fine-tunés : {weights_file}")
                ft_model = load_metric_model(backbone_name, weights_path)
                if ft_model is not None:
                    t_ft_start  = time.perf_counter()
                    embeddings  = extract_embeddings_gpu(
                        ft_model, image_paths, img_size, batch_size,
                        desc=f"{desc_name} (fine-tuné)"
                    )
                    ft_time    = time.perf_counter() - t_ft_start
                    model_type = "finetuned"
                    print(f"  Extraction fine-tuné : {format_time(ft_time)}  → shape {embeddings.shape}")
                    del ft_model
                    torch.cuda.empty_cache()
            else:
                print(f"  ⚠ Fichier introuvable : {weights_path} — utilisation backbone pretrained")

        # PCA + whitening depuis zéro
        print(f"  PCA {embeddings.shape[1]}D → {pca_k}D (fitting sur {N_IMAGES} vecteurs)...")
        t_pca_start = time.perf_counter()
        mean, comps, scale = fit_pca_whitening(embeddings, pca_k)
        reduced = apply_pca_whitening(embeddings, mean, comps, scale)
        pca_time = time.perf_counter() - t_pca_start

        # Temps total : extraction effective + PCA
        eff_extract_time = ft_time if model_type == "finetuned" else extract_time
        total_time       = eff_extract_time + pca_time

        print(f"  PCA time  : {format_time(pca_time)}")
        print(f"  ✓ TOTAL   : {format_time(total_time)}  "
              f"(extract {format_time(eff_extract_time)} + pca {format_time(pca_time)})")
        print(f"  Index final : {reduced.shape}")

        timing_results[desc_name] = {
            "indexing_time_s":    round(total_time, 2),
            "extraction_time_s":  round(eff_extract_time, 2),
            "pca_time_s":         round(pca_time, 2),
            "descriptor_dim":     pca_k,
            "original_dim":       int(embeddings.shape[1]),
            "num_images":         N_IMAGES,
            "device":             f"cuda:{torch.cuda.get_device_name(0)}" if device.type == "cuda" else "cpu",
            "model_type":         model_type,
            "backbone":           backbone_name,
        }

    del backbone
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Benchmark DL terminé.")

## 3. Résultats — Sauvegarde et visualisation

In [ ]:
# ── Sauvegarde JSON sur Drive ─────────────────────────────────────────────────
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

output = {
    "timestamp":  timestamp,
    "device":     f"cuda:{torch.cuda.get_device_name(0)}" if device.type == "cuda" else "cpu",
    "n_images":   N_IMAGES,
    "descriptors": timing_results,
}

json_path = os.path.join(SAVE_DIR, f"indexing_times_{timestamp}.json")
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"JSON sauvegardé : {json_path}")
print("\nRésumé :")
print(f"{'Descripteur':<25} {'Extraction':>12} {'PCA':>8} {'TOTAL':>10}")
print("-" * 58)
for name, r in timing_results.items():
    ext = format_time(r['extraction_time_s'])
    pca = format_time(r['pca_time_s'])
    tot = format_time(r['indexing_time_s'])
    print(f"  {name:<23} {ext:>12} {pca:>8} {tot:>10}")

In [ ]:
# ── Graphique 1 : Temps d'indexation total (échelle log) ─────────────────────
LABELS = {
    "color_histogram":    "Histo. Couleur",
    "sift":               "SIFT",
    "mobilenet_arcface":  "MobileNet ArcFace",
    "mobilenet_zeroshot": "MobileNet ZeroShot",
    "resnet50_zeroshot":  "ResNet50 ZeroShot",
    "vit_b16_zeroshot":   "ViT-B/16 ZeroShot",
    "dinov2_supcon":      "DinoV2 SupCon",
    "dinov2_zeroshot":    "DinoV2 ZeroShot",
}
COLORS = {
    "color_histogram":    "#8b5cf6",
    "sift":               "#a78bfa",
    "mobilenet_arcface":  "#fb923c",
    "mobilenet_zeroshot": "#fcd34d",
    "resnet50_zeroshot":  "#34d399",
    "vit_b16_zeroshot":   "#f9a8d4",
    "dinov2_supcon":      "#818cf8",
    "dinov2_zeroshot":    "#a5b4fc",
}

names  = list(timing_results.keys())
times  = [timing_results[n]["indexing_time_s"] for n in names]
labels = [LABELS.get(n, n) for n in names]
colors = [COLORS.get(n, "#94a3b8") for n in names]

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(range(len(names)), times, color=colors, edgecolor="white", linewidth=0.5)

# Annotations de valeur
for bar, t in zip(bars, times):
    ax.text(
        t * 1.05, bar.get_y() + bar.get_height() / 2,
        format_time(t), va="center", ha="left", fontsize=10, fontweight="bold"
    )

ax.set_xscale("log")
ax.set_yticks(range(len(names)))
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel("Temps d'indexation (s) — échelle logarithmique", fontsize=12)
ax.set_title(
    f"Temps d'indexation des descripteurs MIR\n"
    f"({N_IMAGES} images · GPU {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'})",
    fontsize=13, fontweight="bold"
)
ax.grid(axis="x", which="both", linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

# Séparateur classique / DL
n_classical = sum(1 for n in names if n in ("color_histogram", "sift"))
if n_classical > 0:
    ax.axhline(n_classical - 0.5, color="#475569", linestyle=":", linewidth=1.5, alpha=0.7)
    ax.text(min(times) * 0.5, n_classical - 0.5,
            " Deep Learning ▶ ", va="center", ha="left", fontsize=9,
            color="#475569", style="italic")

plt.tight_layout()

fig1_path = os.path.join(SAVE_DIR, f"indexing_times_bar_{timestamp}.png")
plt.savefig(fig1_path, dpi=200, bbox_inches="tight", facecolor="white")
print(f"Graphique 1 sauvegardé : {fig1_path}")
plt.show()

In [ ]:
# ── Graphique 2 : Barres empilées Extraction vs PCA ───────────────────────────
dl_names = [n for n in names if n not in ("color_histogram", "sift")]

if dl_names:
    ext_times = [timing_results[n]["extraction_time_s"] for n in dl_names]
    pca_times = [timing_results[n]["pca_time_s"]        for n in dl_names]
    dl_labels = [LABELS.get(n, n) for n in dl_names]

    x = np.arange(len(dl_names))
    width = 0.55

    fig, ax = plt.subplots(figsize=(12, 5))

    p1 = ax.bar(x, ext_times, width, label="Extraction (inférence)", color="#6366f1", alpha=0.9)
    p2 = ax.bar(x, pca_times, width, bottom=ext_times, label="PCA + whitening + L2-norm",
                color="#f59e0b", alpha=0.9)

    for i, (ext, pca) in enumerate(zip(ext_times, pca_times)):
        total = ext + pca
        ax.text(i, total + total * 0.03, format_time(total),
                ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(dl_labels, rotation=20, ha="right", fontsize=10)
    ax.set_ylabel("Temps (s)", fontsize=12)
    ax.set_title(
        "Décomposition du temps d'indexation : Extraction vs PCA\n"
        f"(GPU {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'} · {N_IMAGES} images)",
        fontsize=12, fontweight="bold"
    )
    ax.legend(fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)

    plt.tight_layout()
    fig2_path = os.path.join(SAVE_DIR, f"indexing_breakdown_{timestamp}.png")
    plt.savefig(fig2_path, dpi=200, bbox_inches="tight", facecolor="white")
    print(f"Graphique 2 sauvegardé : {fig2_path}")
    plt.show()

In [ ]:
# ── Tableau récapitulatif ─────────────────────────────────────────────────────
print("\n" + "═" * 75)
print(" RÉSULTATS FINAUX — TEMPS D'INDEXATION SUR T4 GPU")
print("═" * 75)
print(f" {'Descripteur':<25} {'Dim':>6} {'Extraction':>12} {'PCA':>10} {'TOTAL':>10} {'Modèle':>12}")
print("-" * 75)

for name, r in timing_results.items():
    dim  = f"{r['original_dim']}→{r['descriptor_dim']}" if r['original_dim'] else str(r['descriptor_dim'])
    ext  = format_time(r['extraction_time_s'])
    pca  = format_time(r['pca_time_s'])
    tot  = format_time(r['indexing_time_s'])
    mtype = r.get('model_type', 'classical')
    print(f"  {name:<23} {dim:>8} {ext:>12} {pca:>10} {tot:>10} {mtype:>12}")

print("═" * 75)
print(f" Sauvegardé dans : {SAVE_DIR}")
print(f" JSON : {os.path.basename(json_path)}")
print("═" * 75)

## 4. Mise à jour de `indexes/metrics.json` (optionnel)

Exécutez cette cellule pour mettre à jour le fichier `metrics.json` du projet avec les temps mesurés sur T4.

In [ ]:
# ── Mise à jour optionnelle de indexes/metrics.json ───────────────────────────
metrics_path = os.path.join(DRIVE_ROOT, "indexes", "metrics.json")

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)

    updated = []
    for name, r in timing_results.items():
        if name in metrics:
            metrics[name]["indexing_time_s"] = r["indexing_time_s"]
            metrics[name]["source"] = "precomputed"  # mesuré sur Colab
            updated.append(name)

    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"metrics.json mis à jour : {updated}")
    print(f"Fichier : {metrics_path}")
else:
    print(f"⚠ metrics.json introuvable : {metrics_path}")
    print("  Créez-le manuellement ou ajustez DRIVE_ROOT.")